# MoE Router: From Scratch

The **Router** is the brain of a Mixture of Experts model. It decides which experts process which tokens.

In this tutorial, we will:
1. Build a router step-by-step.
2. Understand Top-K gating.
3. Implement the noise mechanism for training stability.

Explained in detail here - https://youtu.be/O2yAMJu8LpI

## 1. The Gating Mechanism

The router is just a linear layer that projects the input `d_model` to `num_experts` scores.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

d_model = 16
num_experts = 4

# The gate
gate = nn.Linear(d_model, num_experts, bias=False)

# Dummy input [batch, seq, d_model]
x = torch.randn(1, 5, d_model)

# Compute logits
logits = gate(x)
print("Logits shape:", logits.shape) # [1, 5, 4]

## 2. Top-K Selection

We don't want to use all experts. We only want the top $k$ (usually 2). This makes the model sparse and fast.

In [ ]:
top_k = 2

# Find top-k values and their indices
top_k_logits, indices = torch.topk(logits, top_k, dim=-1)

print("Top-k Indices:", indices)
print("Top-k Logits:", top_k_logits)

## 3. Normalization

We need the weights to sum to 1, so we apply Softmax to the top-k logits.

In [ ]:
weights = F.softmax(top_k_logits, dim=-1)
print("Weights:", weights)

## 4. Full Class Implementation

Let's wrap this in a clean PyTorch module, adding noise for training (which helps prevent expert collapse).

In [ ]:
class TopKRouter(nn.Module):
    def __init__(self, d_model, num_experts, top_k=2):
        super().__init__()
        self.gate = nn.Linear(d_model, num_experts, bias=False)
        self.top_k = top_k
        self.noise_std = 0.1

    def forward(self, x):
        logits = self.gate(x)
        
        if self.training:
            noise = torch.randn_like(logits) * self.noise_std
            logits = logits + noise
            
        top_k_logits, indices = torch.topk(logits, self.top_k, dim=-1)
        weights = F.softmax(top_k_logits, dim=-1)
        
        return weights, indices

router = TopKRouter(d_model, num_experts)
w, i = router(x)
print("Final Weights:", w)